# Modeling - UNSW-NB15

Questo notebook usa i dataset gia preprocessati nel notebook `02_preprocessing.ipynb` e implementa la fase di modellazione per il task principale di classificazione binaria (`label`: traffico normale vs attacco).

Principi metodologici:
- il test set ufficiale viene usato solo per la valutazione finale;
- la selezione degli iperparametri avviene solo sul training set tramite Stratified K-Fold Cross-Validation;
- lo scaling viene inserito dentro le pipeline sklearn, quindi viene fittato solo sul training fold durante la cross-validation;
- `attack_cat` viene esclusa dalle feature del task binario per evitare leakage.


In [ ]:
# ==========================================
# 1. IMPORT E CONFIGURAZIONE AMBIENTE
# ==========================================
import logging
import time
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    calinski_harabasz_score,
    classification_report,
    confusion_matrix,
    davies_bouldin_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    silhouette_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.svm import LinearSVC

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    XGBClassifier = None

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s", force=True)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
N_JOBS = 6

print("Ambiente configurato.")
print(f"XGBoost disponibile: {XGBOOST_AVAILABLE}")


## Path del Progetto

I due dataset usati in questa fase sono quelli salvati dal notebook di preprocessing nella cartella `processed_data`.


In [ ]:
# ==========================================
# 2. MOUNT DRIVE E DEFINIZIONE PATH
# ==========================================
if IN_COLAB and Path("/var/colab/hostname").exists():
    if not Path("/content/drive/MyDrive").exists():
        logging.info("Montaggio di Google Drive in corso...")
        drive.mount("/content/drive")
    else:
        logging.info("Google Drive gia montato.")
    BASE_PATH = Path("/content/drive/MyDrive/progetto FDSL")
else:
    logging.info("Local runtime / ambiente non-Colab hosted rilevato: uso path locale.")
    BASE_PATH = Path("/content/progetto FDSL")

PROCESSED_DATA_PATH = BASE_PATH / "processed_data"
TRAIN_PROCESSED_PATH = PROCESSED_DATA_PATH / "UNSW_NB15_training-set_preprocessed.parquet"
TEST_PROCESSED_PATH = PROCESSED_DATA_PATH / "UNSW_NB15_testing-set_preprocessed.parquet"

REPO_PATH = BASE_PATH / "unsw-nb15-network-ids"

RESULTS_PATH = REPO_PATH / "results" / "modeling"
TABLES_PATH = RESULTS_PATH / "tables"
PLOTS_PATH = RESULTS_PATH / "plots"
MODELS_PATH = RESULTS_PATH / "models"

for path in [TABLES_PATH, PLOTS_PATH, MODELS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Path configurati:")
print(f"TRAIN: {TRAIN_PROCESSED_PATH}")
print(f"TEST : {TEST_PROCESSED_PATH}")
print(f"OUT  : {RESULTS_PATH}")

## Caricamento Dataset e Separazione X/y

La variabile `label` è il target del task binario. La variabile `attack_cat` viene preservata nel dataset preprocessato, ma viene esclusa dalle feature per evitare leakage nel task binario.


In [ ]:
# ==========================================
# 3. CARICAMENTO DATASET E CONTROLLI
# ==========================================
TARGET = "label"
LEAKAGE_COLUMNS = ["attack_cat"]


def load_parquet_dataset(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"File non trovato: {path}")
    df_loaded = pd.read_parquet(path)
    logging.info(f"Caricato {path.name}: {df_loaded.shape[0]} righe, {df_loaded.shape[1]} colonne")
    return df_loaded


def split_features_target(
    df: pd.DataFrame,
    target: str = TARGET,
    leakage_columns: Optional[List[str]] = None,
) -> Tuple[pd.DataFrame, pd.Series]:
    leakage_columns = leakage_columns or []
    missing = [col for col in [target] + leakage_columns if col not in df.columns]
    if missing:
        raise ValueError(f"Colonne attese mancanti nel dataset: {missing}")

    y = df[target].astype(int)
    X = df.drop(columns=[target] + leakage_columns)
    return X, y


def validate_modeling_matrices(X_train: pd.DataFrame, X_test: pd.DataFrame, y_train: pd.Series, y_test: pd.Series) -> None:
    if list(X_train.columns) != list(X_test.columns):
        train_only = sorted(set(X_train.columns) - set(X_test.columns))
        test_only = sorted(set(X_test.columns) - set(X_train.columns))
        raise ValueError(
            "Train e test non hanno le stesse feature. "
            f"Solo train: {train_only[:10]} | Solo test: {test_only[:10]}"
        )

    if X_train.isna().any().any() or X_test.isna().any().any():
        raise ValueError("Sono presenti valori NaN in X_train o X_test.")

    if sorted(y_train.unique().tolist()) != [0, 1] or sorted(y_test.unique().tolist()) != [0, 1]:
        raise ValueError("Il target label non risulta binario con valori 0/1.")

    logging.info("Controlli superati: feature allineate, nessun NaN, target binario valido.")


df_train = load_parquet_dataset(TRAIN_PROCESSED_PATH)
df_test = load_parquet_dataset(TEST_PROCESSED_PATH)

X_train, y_train = split_features_target(df_train, leakage_columns=LEAKAGE_COLUMNS)
X_test, y_test = split_features_target(df_test, leakage_columns=LEAKAGE_COLUMNS)

validate_modeling_matrices(X_train, X_test, y_train, y_test)

feature_names = X_train.columns.tolist()

print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print("Distribuzione y_train:")
display(y_train.value_counts(normalize=False).rename("count").to_frame().assign(percent=y_train.value_counts(normalize=True).mul(100).round(2)))
print("Distribuzione y_test:")
display(y_test.value_counts(normalize=False).rename("count").to_frame().assign(percent=y_test.value_counts(normalize=True).mul(100).round(2)))


## Cross-Validation e Metriche

La Grid Search viene eseguita solo sul training set con `StratifiedKFold`. Il parametro `refit="f1"` seleziona il modello migliore rispetto all'F1-score, mantenendo comunque traccia di accuracy, precision, recall, ROC-AUC e PR-AUC.


In [ ]:
# ==========================================
# 4. CROSS-VALIDATION, METRICHE E UTILITY
# ==========================================
CV_SPLITTER = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

SCORING = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

REFIT_METRIC = "f1"


def safe_name(name: str) -> str:
    return name.lower().replace(" ", "_").replace("-", "_")


def get_positive_scores(model, X: pd.DataFrame) -> Optional[np.ndarray]:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return None


def evaluate_binary_classifier(y_true: pd.Series, y_pred: np.ndarray, y_scores: Optional[np.ndarray]) -> Dict[str, float]:
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    if y_scores is not None:
        metrics["roc_auc"] = roc_auc_score(y_true, y_scores)
        metrics["pr_auc"] = average_precision_score(y_true, y_scores)
    else:
        metrics["roc_auc"] = np.nan
        metrics["pr_auc"] = np.nan

    return metrics


def save_cv_results(search: GridSearchCV, model_name: str) -> pd.DataFrame:
    cv_results = pd.DataFrame(search.cv_results_)
    out_file = TABLES_PATH / f"{safe_name(model_name)}_cv_results.csv"
    cv_results.to_csv(out_file, index=False)
    logging.info(f"Risultati CV salvati in: {out_file}")
    return cv_results


## Grafici e Salvataggio

Le funzioni seguenti salvano confusion matrix, curve ROC/PR e feature importance quando disponibili.


In [ ]:
# ==========================================
# 5. GRAFICI E SALVATAGGIO OUTPUT
# ==========================================
def plot_confusion_matrix_file(y_true, y_pred, model_name: str) -> None:
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.savefig(PLOTS_PATH / f"{safe_name(model_name)}_confusion_matrix.png", dpi=150)
    plt.close()


def plot_roc_pr_curves(y_true, y_scores, model_name: str) -> None:
    if y_scores is None:
        return

    fpr, tpr, _ = roc_curve(y_true, y_scores)
    precision, recall, _ = precision_recall_curve(y_true, y_scores)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    axes[0].plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_true, y_scores):.4f}")
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
    axes[0].set_title(f"ROC Curve - {model_name}")
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].legend(loc="lower right")

    axes[1].plot(recall, precision, label=f"PR-AUC = {average_precision_score(y_true, y_scores):.4f}")
    axes[1].set_title(f"Precision-Recall Curve - {model_name}")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].legend(loc="lower left")

    plt.tight_layout()
    plt.savefig(PLOTS_PATH / f"{safe_name(model_name)}_roc_pr_curves.png", dpi=150)
    plt.close()


def get_feature_importance(best_estimator, feature_names: List[str]) -> Optional[pd.DataFrame]:
    model = best_estimator.named_steps.get("model") if isinstance(best_estimator, Pipeline) else best_estimator

    if hasattr(model, "feature_importances_"):
        values = model.feature_importances_
    elif hasattr(model, "coef_"):
        values = np.abs(np.ravel(model.coef_))
    else:
        return None

    importance_df = pd.DataFrame({"feature": feature_names, "importance": values})
    importance_df = importance_df.sort_values("importance", ascending=False).reset_index(drop=True)
    return importance_df


def save_and_plot_feature_importance(best_estimator, feature_names: List[str], model_name: str) -> None:
    importance_df = get_feature_importance(best_estimator, feature_names)
    if importance_df is None:
        return

    importance_df.to_csv(TABLES_PATH / f"{safe_name(model_name)}_feature_importance.csv", index=False)
    top = importance_df.head(20).iloc[::-1]

    plt.figure(figsize=(9, 6))
    plt.barh(top["feature"], top["importance"])
    plt.title(f"Top 20 Feature Importance - {model_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.savefig(PLOTS_PATH / f"{safe_name(model_name)}_feature_importance.png", dpi=150)
    plt.close()


## Definizione dei Modelli Supervisionati

Ogni modello e definito come pipeline o stimatore sklearn. Logistic Regression e Linear SVM includono `RobustScaler` dentro la pipeline; Random Forest, Extra Trees e XGBoost non richiedono scaling.


In [ ]:
# ==========================================
# 6. MODELLI SUPERVISIONATI E GRIGLIE PARAMETRICHE
# ==========================================
def build_model_specs() -> Dict[str, Dict]:
    specs = {
        "Logistic Regression": {
            "estimator": Pipeline([
                ("scaler", RobustScaler()),
                ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, solver="liblinear")),
            ]),
            "param_grid": {
                "model__C": [0.1, 1.0, 10.0],
                "model__penalty": ["l1", "l2"],
                "model__class_weight": [None, "balanced"],
            },
        },
        "Linear SVM": {
            "estimator": Pipeline([
                ("scaler", RobustScaler()),
                ("model", LinearSVC(max_iter=1000, random_state=RANDOM_STATE, dual=False)),
            ]),
            "param_grid": {
                "model__C": [0.1, 1.0, 10.0],
                "model__class_weight": [None, "balanced"],
            },
        },
        "Random Forest": {
            "estimator": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
            "param_grid": {
                "n_estimators": [100, 200],
                "max_depth": [None, 20],
                "min_samples_leaf": [1, 2],
                "class_weight": [None, "balanced"],
            },
        },
        "Extra Trees": {
            "estimator": ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=1),
            "param_grid": {
                "n_estimators": [100, 200],
                "max_depth": [None, 20],
                "min_samples_leaf": [1, 2],
                "class_weight": [None, "balanced"],
            },
        },
    }

    if XGBOOST_AVAILABLE:
        scale_pos_weight = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
        specs["XGBoost"] = {
            "estimator": XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=1,
                scale_pos_weight=scale_pos_weight,
            ),
            "param_grid": {
                "n_estimators": [100, 200],
                "max_depth": [3, 6],
                "learning_rate": [0.05, 0.1],
                "subsample": [0.8, 1.0],
                "colsample_bytree": [0.8, 1.0],
            },
        }
    else:
        logging.warning("XGBoost non disponibile: il modello verra saltato.")

    return specs


model_specs = build_model_specs()
print("Modelli supervisionati configurati:")
for model_name, spec in model_specs.items():
    grid_size = 1
    for values in spec["param_grid"].values():
        grid_size *= len(values)
    print(f"- {model_name}: {grid_size} combinazioni")


## Grid Search su Training Set

Questa cella puo richiedere tempo. Il test set non viene usato durante la Grid Search.


In [ ]:
# ==========================================
# 7. GRID SEARCH E VALUTAZIONE FINALE SUPERVISIONATA
# ==========================================
from joblib import parallel_backend

def run_grid_search_for_model(model_name: str, estimator, param_grid: Dict) -> GridSearchCV:
    logging.info(f"Avvio GridSearchCV: {model_name}")
    search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring=SCORING,
        refit=REFIT_METRIC,
        cv=CV_SPLITTER,
        n_jobs=N_JOBS,
        verbose=3,
        return_train_score=True,
    )
    with parallel_backend("loky", inner_max_num_threads=1):
        search.fit(X_train, y_train)
    logging.info(f"{model_name} completato. Best {REFIT_METRIC}: {search.best_score_:.4f}")
    logging.info(f"Best params: {search.best_params_}")
    save_cv_results(search, model_name)
    return search

def evaluate_best_model_on_test(model_name: str, best_estimator) -> Dict:
    start_time = time.time()
    y_pred = best_estimator.predict(X_test)
    inference_time = time.time() - start_time
    y_scores = get_positive_scores(best_estimator, X_test)

    metrics = evaluate_binary_classifier(y_test, y_pred, y_scores)
    report = classification_report(y_test, y_pred, zero_division=0, output_dict=True)
    pd.DataFrame(report).transpose().to_csv(TABLES_PATH / f"{safe_name(model_name)}_classification_report.csv")

    plot_confusion_matrix_file(y_test, y_pred, model_name)
    plot_roc_pr_curves(y_test, y_scores, model_name)
    save_and_plot_feature_importance(best_estimator, feature_names, model_name)

    model_file = MODELS_PATH / f"{safe_name(model_name)}_best_model.joblib"
    joblib.dump(best_estimator, model_file)

    return {
        "model": model_name,
        **metrics,
        "inference_time_seconds": inference_time,
        "model_path": str(model_file),
    }


supervised_results = []
best_searches = {}

for model_name, spec in model_specs.items():
    start_time = time.time()
    search = run_grid_search_for_model(model_name, spec["estimator"], spec["param_grid"])
    training_time = time.time() - start_time

    best_searches[model_name] = search
    result = evaluate_best_model_on_test(model_name, search.best_estimator_)
    result["best_cv_f1"] = search.best_score_
    result["training_time_seconds"] = training_time
    result["best_params"] = str(search.best_params_)
    supervised_results.append(result)

supervised_results_df = pd.DataFrame(supervised_results).sort_values("f1", ascending=False).reset_index(drop=True)
supervised_results_df.to_csv(TABLES_PATH / "supervised_test_results.csv", index=False)

display(supervised_results_df)


## Esperimento Non Supervisionato: KMeans

KMeans viene trattato come analisi esplorativa separata. Si definiscono due scenari: caso binario con esclusione della variabile target `attack_cat`; caso multiclasse con esclusione della variabile target label. Quindi si esegue l'algortimo di K-Means con numero di cluster variabile da `k=2` a `k=10` per verificare se la struttura dei dati tende a separare traffico normale e traffico di attacco. Le etichette reali vengono usate solo a posteriori per interpretare i cluster.


In [ ]:
# ==========================================
# CARICAMENTO DATASET PREPROCESSATI
# ==========================================

train_df_kmeans = load_parquet_dataset(TRAIN_PROCESSED_PATH)
test_df_kmeans = load_parquet_dataset(TEST_PROCESSED_PATH)

In [ ]:
# ==========================================
# 8. KMEANS COME ESPERIMENTO NON SUPERVISIONATO
# ==========================================

def map_clusters_to_majority_label(cluster_labels: np.ndarray, y_reference: pd.Series) -> Dict[int, int]:
    mapping = {}
    for cluster_id in np.unique(cluster_labels):
        mask = cluster_labels == cluster_id
        majority_label = y_reference[mask].mode().iloc[0]
        mapping[int(cluster_id)] = majority_label
    return mapping


def evaluate_multiclass_classifier(y_true: pd.Series, y_pred: np.ndarray) -> Dict:
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


def plot_confusion_matrix_multiclass(y_true: pd.Series, y_pred: np.ndarray, title: str, filename: str) -> None:
    from sklearn.metrics import confusion_matrix

    labels = sorted(pd.unique(pd.concat([pd.Series(y_true), pd.Series(y_pred)])))
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    plt.figure(figsize=(8, 6.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(PLOTS_PATH / filename, dpi=150)
    plt.close()


def run_kmeans_experiment(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series,
    k: int,
    experiment_tag: str,
    target_col: str,
) -> Dict:
    kmeans_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)),
    ])

    start_time = time.time()
    kmeans_pipeline.fit(X_train)
    training_time = time.time() - start_time

    train_clusters = kmeans_pipeline.predict(X_train)
    test_clusters = kmeans_pipeline.predict(X_test)

    cluster_mapping = map_clusters_to_majority_label(train_clusters, y_train.reset_index(drop=True))
    mapped_test_labels = np.array([cluster_mapping[int(c)] for c in test_clusters])

    scaler = kmeans_pipeline.named_steps["scaler"]
    X_test_scaled = scaler.transform(X_test)
    sample_size = min(10000, X_test_scaled.shape[0])

    cluster_metrics = {
        "silhouette_score": silhouette_score(X_test_scaled, test_clusters, sample_size=sample_size, random_state=RANDOM_STATE),
        "davies_bouldin_score": davies_bouldin_score(X_test_scaled, test_clusters),
        "calinski_harabasz_score": calinski_harabasz_score(X_test_scaled, test_clusters),
    }

    is_multiclass = target_col == "attack_cat"

    if is_multiclass:
        external_metrics = evaluate_multiclass_classifier(y_test, mapped_test_labels)
        plot_confusion_matrix_multiclass(
            y_test, mapped_test_labels,
            title=f"KMeans mapped clusters (k={k}, {experiment_tag})",
            filename=f"kmeans_confusion_{experiment_tag}_k{k}.png",
        )
    else:
        external_metrics = evaluate_binary_classifier(y_test, mapped_test_labels, None)
        plot_confusion_matrix_file(y_test, mapped_test_labels, f"KMeans mapped clusters (k={k}, {experiment_tag})")

    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    X_test_pca = pca.fit_transform(X_test_scaled)
    pca_sample = min(20000, X_test_pca.shape[0])
    rng = np.random.default_rng(RANDOM_STATE)
    sample_idx = rng.choice(X_test_pca.shape[0], size=pca_sample, replace=False)

    plt.figure(figsize=(7, 5.5))
    sns.scatterplot(
        x=X_test_pca[sample_idx, 0], y=X_test_pca[sample_idx, 1],
        hue=test_clusters[sample_idx], palette="Set2", s=12, linewidth=0,
    )
    plt.title(f"KMeans clusters visualizzati su PCA (k={k}, {experiment_tag})")
    plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend(title="Cluster")
    plt.tight_layout()
    plt.savefig(PLOTS_PATH / f"kmeans_pca_clusters_{experiment_tag}_k{k}.png", dpi=150)
    plt.close()

    # Visualizzazione della stessa PCA con le etichette reali
    if is_multiclass:
        label_names = y_test.astype(str)
        palette = "tab10"
        hue_order = sorted(label_names.unique())
    else:
        label_names = y_test.map({0: "Normal", 1: "Attack"})
        palette = {"Normal": "tab:blue", "Attack": "tab:red"}
        hue_order = ["Normal", "Attack"]

    plt.figure(figsize=(7, 5.5))
    sns.scatterplot(
        x=X_test_pca[sample_idx, 0], y=X_test_pca[sample_idx, 1],
        hue=label_names.iloc[sample_idx], hue_order=hue_order,
        palette=palette, s=12, linewidth=0,
    )
    plt.title(f"Etichette reali visualizzate su PCA (k={k}, {experiment_tag})")
    plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend(title="Label")
    plt.tight_layout()
    plt.savefig(PLOTS_PATH / f"kmeans_pca_true_labels_{experiment_tag}_k{k}.png", dpi=150)
    plt.close()

    joblib.dump(kmeans_pipeline, MODELS_PATH / f"kmeans_pipeline_{experiment_tag}_k{k}.joblib")

    return {
        "experiment": experiment_tag,
        "target_col": target_col,
        "model": "KMeans",
        "k": k,
        "cluster_mapping": str(cluster_mapping),
        "training_time_seconds": training_time,
        **cluster_metrics,
        **{f"mapped_{key}": value for key, value in external_metrics.items()},
    }


# -------------------------------------------------------------
# 8.1 Ricaricamento dataset e gestione manuale delle esclusioni
# -------------------------------------------------------------
df_train_full = load_parquet_dataset(TRAIN_PROCESSED_PATH)
df_test_full = load_parquet_dataset(TEST_PROCESSED_PATH)

exclusion_scenarios = {
    "excl_attack_cat": {"drop_col": "attack_cat", "target_col": "label"},
    "excl_label": {"drop_col": "label", "target_col": "attack_cat"},
}

kmeans_results_all = []
kmeans_scenario_dfs = {}

for scenario_name, cfg in exclusion_scenarios.items():
    drop_col, target_col = cfg["drop_col"], cfg["target_col"]

    X_train_manual = df_train_full.drop(columns=[drop_col, target_col])
    y_train_manual = df_train_full[target_col]

    X_test_manual = df_test_full.drop(columns=[drop_col, target_col])
    y_test_manual = df_test_full[target_col]

    scenario_results = []
    for k in range(2, 11):
        res = run_kmeans_experiment(
            X_train_manual, X_test_manual, y_train_manual, y_test_manual,
            k=k, experiment_tag=scenario_name, target_col=target_col,
        )
        scenario_results.append(res)

    kmeans_results_all.extend(scenario_results)
    scenario_df = pd.DataFrame(scenario_results)
    kmeans_scenario_dfs[scenario_name] = scenario_df
    scenario_df.to_csv(TABLES_PATH / f"kmeans_results_{scenario_name}.csv", index=False)

kmeans_results_df = pd.DataFrame(kmeans_results_all)
kmeans_results_df.to_csv(TABLES_PATH / "kmeans_results.csv", index=False)
display(kmeans_results_df)

## Riepilogo Finale

Questa cella mostra i risultati principali salvati su Drive.


In [ ]:
# ==========================================
# 9. RIEPILOGO OUTPUT
# ==========================================
print("File principali salvati:")
print(f"- Risultati supervisionati: {TABLES_PATH / 'supervised_test_results.csv'}")
print(f"- Risultati KMeans: {TABLES_PATH / 'kmeans_results.csv'}")
print(f"- Modelli: {MODELS_PATH}")
print(f"- Grafici: {PLOTS_PATH}")

if 'supervised_results_df' in globals():
    print()
    print("Classifica modelli supervisionati per F1 sul test set:")
    display(supervised_results_df[["model", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "best_cv_f1"]])


## Feature Selection

In [ ]:
# ==========================================================
# FEATURE SELECTION CON XGBOOST + CONFRONTO TRA SOGLIE
# ==========================================================

from pathlib import Path
import time
import joblib
import pandas as pd

from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

from xgboost import XGBClassifier

MODEL_PATH = Path(
    "/content/drive/MyDrive/progetto FDSL/unsw-nb15-network-ids/results/modeling/models/xgboost_best_model.joblib"
)

# ==========================================================
# CONFIGURAZIONE
# ==========================================================

THRESHOLDS = [
    "mean",
    "median",
    "1.25*mean",
    "1.5*mean",
    "1.75*mean",
    "2*mean",
]

N_RUNS = 20

# ==========================================================
# CARICAMENTO MODELLO
# ==========================================================

best_model = joblib.load(MODEL_PATH)
params = best_model.get_params()

all_results = []

# ==========================================================
# BASELINE: MODELLO ORIGINALE SU TUTTE LE FEATURE (X_test)
# ==========================================================

print("\n" + "=" * 80)
print("Baseline (modello originale, tutte le feature)")
print("=" * 80)

y_pred_baseline = best_model.predict(X_test)
y_prob_baseline = best_model.predict_proba(X_test)[:, 1]

# ------------------------------------------------------
# Tempo medio di inferenza (20 esecuzioni) - stesso identico
# procedimento usato piu' avanti per ogni soglia
# ------------------------------------------------------

inference_times_baseline = []

for _ in range(N_RUNS):
    start_time = time.perf_counter()
    _ = best_model.predict(X_test)
    inference_times_baseline.append(time.perf_counter() - start_time)

inference_time_baseline = sum(inference_times_baseline) / N_RUNS

# ------------------------------------------------------
# Metriche baseline (stesse funzioni/stesso modo di calcolo
# usate nel ciclo sulle soglie)
# ------------------------------------------------------

baseline_result = {
    "threshold": "baseline (all features)",
    "n_features": X_test.shape[1],
    "accuracy": accuracy_score(y_test, y_pred_baseline),
    "precision": precision_score(y_test, y_pred_baseline, zero_division=0),
    "recall": recall_score(y_test, y_pred_baseline, zero_division=0),
    "f1": f1_score(y_test, y_pred_baseline, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_prob_baseline),
    "pr_auc": average_precision_score(y_test, y_prob_baseline),
    "inference_time_seconds": inference_time_baseline,
}

print(f"Feature usate: {baseline_result['n_features']}")

all_results.append(baseline_result)

# ==========================================================
# CICLO SULLE SOGLIE (invariato)
# ==========================================================

for threshold in THRESHOLDS:

    print("\n" + "=" * 80)
    print(f"Threshold: {threshold}")
    print("=" * 80)

    # ------------------------------------------------------
    # Feature Selection (nessun nuovo training)
    # ------------------------------------------------------

    selector = SelectFromModel(
        estimator=best_model,
        prefit=True,
        threshold=threshold,
    )

    selected_features = X_train.columns[selector.get_support()].tolist()

    print(f"Feature selezionate: {len(selected_features)}/{X_train.shape[1]}")

    X_train_fs = X_train[selected_features].copy()
    X_test_fs = X_test[selected_features].copy()

    # ------------------------------------------------------
    # Nuovo XGBoost con gli stessi iperparametri
    # ------------------------------------------------------

    model = XGBClassifier(**params)

    model.fit(X_train_fs, y_train)

    # ------------------------------------------------------
    # Predizioni per le metriche
    # ------------------------------------------------------

    y_pred = model.predict(X_test_fs)
    y_prob = model.predict_proba(X_test_fs)[:, 1]

    # ------------------------------------------------------
    # Tempo medio di inferenza (20 esecuzioni)
    # ------------------------------------------------------

    inference_times = []

    for _ in range(N_RUNS):
        start_time = time.perf_counter()
        _ = model.predict(X_test_fs)
        inference_times.append(time.perf_counter() - start_time)

    inference_time = sum(inference_times) / N_RUNS

    # ------------------------------------------------------
    # Metriche
    # ------------------------------------------------------

    all_results.append({
        "threshold": threshold,
        "n_features": len(selected_features),
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "pr_auc": average_precision_score(y_test, y_prob),
        "inference_time_seconds": inference_time,
    })

# ==========================================================
# RISULTATI FINALI
# ==========================================================

results_df = pd.DataFrame(all_results)

print("\n")
print("=" * 130)
print("RISULTATI FEATURE SELECTION")
print("=" * 130)

print(
    results_df.round({
        "accuracy": 6,
        "precision": 6,
        "recall": 6,
        "f1": 6,
        "roc_auc": 6,
        "pr_auc": 6,
        "inference_time_seconds": 6,
    }).to_string(index=False)
)

# ==========================================================
# CONFRONTO CON LA BASELINE (incrementi/decrementi per soglia)
# ==========================================================

comparison_rows = []

for _, row in results_df.iterrows():
    if row["threshold"] == "baseline (all features)":
        continue

    delta_inference_time = row["inference_time_seconds"] - baseline_result["inference_time_seconds"]
    pct_inference_time = (
        (delta_inference_time / baseline_result["inference_time_seconds"]) * 100
        if baseline_result["inference_time_seconds"] != 0
        else float("nan")
    )

    delta_f1 = row["f1"] - baseline_result["f1"]
    pct_f1 = (
        (delta_f1 / baseline_result["f1"]) * 100
        if baseline_result["f1"] != 0
        else float("nan")
    )

    delta_roc_auc = row["roc_auc"] - baseline_result["roc_auc"]
    pct_roc_auc = (
        (delta_roc_auc / baseline_result["roc_auc"]) * 100
        if baseline_result["roc_auc"] != 0
        else float("nan")
    )

    comparison_rows.append({
        "threshold": row["threshold"],
        "n_features": row["n_features"],
        "delta_inference_time_s": delta_inference_time,
        "delta_inference_time_%": pct_inference_time,
        "delta_f1": delta_f1,
        "delta_f1_%": pct_f1,
        "delta_roc_auc": delta_roc_auc,
        "delta_roc_auc_%": pct_roc_auc,
    })

comparison_df = pd.DataFrame(comparison_rows)

print("\n")
print("=" * 130)
print("CONFRONTO CON LA BASELINE (tutte le feature, modello originale)")
print("=" * 130)
print(
    f"Baseline -> F1: {baseline_result['f1']:.6f} | "
    f"ROC-AUC: {baseline_result['roc_auc']:.6f} | "
    f"Inference time: {baseline_result['inference_time_seconds']:.6f}s"
)
print()

print(
    comparison_df.round({
        "delta_inference_time_s": 6,
        "delta_inference_time_%": 2,
        "delta_f1": 6,
        "delta_f1_%": 2,
        "delta_roc_auc": 6,
        "delta_roc_auc_%": 2,
    }).to_string(index=False)
)

In [ ]:
# ==========================================================
# FEATURE SELEZIONATE PER LA SOGLIA "median" + PARAMETRI MODELLO
# ==========================================================

selector_median = SelectFromModel(
    estimator=best_model,
    prefit=True,
    threshold="median",
)

selected_features_median = X_train.columns[selector_median.get_support()].tolist()

print(f"Numero di feature selezionate (median): {len(selected_features_median)}")
print("\nFeature selezionate:")
for feat in selected_features_median:
    print(f"- {feat}")

print("\nParametri del modello:")
for k, v in best_model.get_params().items():
    print(f"{k}: {v}")